In [1]:
pip install matplotlib

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import customtkinter as ctk
import psycopg2
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

# Set global appearance mode
ctk.set_appearance_mode("light")

# Precise Figma Palette hex codes mapping
COLOR_SIDEBAR = "#4B1E0F"       
COLOR_WORKSPACE_BG = "#F3EBDD"   
COLOR_CARD_DARK = "#5A2D0C"      
COLOR_CARD_ORANGE = "#B85C2E"    
COLOR_CARD_MID = "#6B3F1D"       
COLOR_CARD_LIGHT = "#D9C2A3"     
COLOR_TEXT_DARK = "#3E2A1F"      
COLOR_PANEL_BG = "#EADFC9"       # The warm card background color

# DATABASE CONNECTION CORE

def get_db_connection():
    try:
        conn = psycopg2.connect(
            host="localhost", database="safepath", 
            user="postgres", password="cos101", port="5432"
        )
        return conn
    except:
        return None

#DATABASE QUERIES & FALLBACKS

def fetch_dashboard_metrics():

    conn = get_db_connection()
    cur = conn.cursor()

    # TOTAL REPORTS
    cur.execute("SELECT COUNT(*) FROM reports")
    total_reports = cur.fetchone()[0]

    # HIGH RISK REPORTS
    cur.execute("""
        SELECT COUNT(*)
        FROM reports
        WHERE urgency ILIKE 'High'
    """)
    high_risk = cur.fetchone()[0]

    # AWARENESS CHECKS
    # Placeholder logic for now
    awareness = total_reports + 5

    # PEOPLE HELPED
    # Placeholder estimation
    helped = total_reports
    
    # RECENT REPORTS
    cur.execute("""
        SELECT incident_type, location
        FROM reports
        ORDER BY id DESC
        LIMIT 5
    """)

    recent_rows = cur.fetchall()

    recent_reports = []

    for incident, location in recent_rows:
        recent_reports.append(
            f"{incident} reported in {location}"
        )

    cur.close()
    conn.close()

    return {
        "total": total_reports,
        "high_risk": high_risk,
        "awareness": awareness,
        "helped": helped,
        "recent": recent_reports
    }
def fetch_insights_data():
    conn = get_db_connection()
    if conn is None:
        return {
            "categories": ["Human Trafficking", "Sexual Exploitation", "Forced Overtime", "Debt Bondage", "Child Labour", "Wage Withholding"],
            "counts": [3, 1, 4, 2, 2, 2],
            "text_insights": [
                "📈 System profiles indicate high concentrations in industrial sectors.",
                "📍 Hotspot analysis: Elevated count of high-severity flags detected."
            ]
        }
    try:
        cursor = conn.cursor()
        cursor.execute("SELECT incident_type, COUNT(*) FROM reports GROUP BY incident_type ORDER BY COUNT(*) DESC;")
        rows = cursor.fetchall()
        categories = [row[0] if row[0] else "Uncategorized" for row in rows]
        counts = [row[1] for row in rows]
        cursor.close(); conn.close()
        return {"categories": categories, "counts": counts, "text_insights": [f"📊 Live tracking {sum(counts)} profiles across all sectors.", "📍 Check Records page for data splits."]}
    except:
        return {"categories": ["SQL Error"], "counts": [1], "text_insights": ["Error loading live trends."]}


# MAIN APPLICATION WINDOW
root = ctk.CTk()
root.geometry("1440x900")
root.title("SafePath System Hub")

# Global UI Content Container
main_frame = ctk.CTkFrame(root, fg_color=COLOR_WORKSPACE_BG, corner_radius=0)


#NAVIGATION MECHANICS

def clear_workspace():
    """Wipes out everything in the workspace frame so the next screen can load smoothly."""
    for widget in main_frame.winfo_children():
        widget.destroy()

def load_dashboard_view():
    clear_workspace()
    update_sidebar_highlights("Dashboard")
    
    # Titles
    ctk.CTkLabel(main_frame, text="Dashboard", font=("Georgia", 38, "bold"), text_color=COLOR_TEXT_DARK).pack(anchor="nw", padx=40, pady=(40, 5))
    ctk.CTkLabel(main_frame, text="Welcome back, James.", font=("Arial", 16), text_color="#5E4B3C").pack(anchor="nw", padx=40, pady=(0, 25))
    
    # Grid Setup
    grid = ctk.CTkFrame(main_frame, fg_color="transparent")
    grid.pack(fill="both", expand=True, padx=40, pady=(0, 40))
    grid.columnconfigure(0, weight=1); grid.columnconfigure(1, weight=1)
    grid.rowconfigure(0, weight=1); grid.rowconfigure(1, weight=1); grid.rowconfigure(2, weight=2)
    
    # Render 4 Cards Matrix
    c1 = ctk.CTkFrame(grid, fg_color=COLOR_CARD_DARK, corner_radius=18); c1.grid(row=0, column=0, sticky="nsew", padx=(0,15), pady=10); c1.pack_propagate(False)
    ctk.CTkLabel(c1, text="Total Reports", font=("Arial", 16), text_color="#F3EBDD").pack(anchor="nw", padx=25, pady=(25,5))
    lbl_total = ctk.CTkLabel(c1, text="--", font=("Georgia", 42, "bold"), text_color="white"); lbl_total.pack(anchor="nw", padx=25)
    
    c2 = ctk.CTkFrame(grid, fg_color=COLOR_CARD_ORANGE, corner_radius=18); c2.grid(row=0, column=1, sticky="nsew", padx=(15,0), pady=10); c2.pack_propagate(False)
    ctk.CTkLabel(c2, text="High Risk Flagged", font=("Arial", 16, "bold"), text_color=COLOR_SIDEBAR).pack(anchor="nw", padx=25, pady=(25,5))
    lbl_highrisk = ctk.CTkLabel(c2, text="--", font=("Georgia", 42, "bold"), text_color=COLOR_SIDEBAR); lbl_highrisk.pack(anchor="nw", padx=25)
    
    c3 = ctk.CTkFrame(grid, fg_color=COLOR_CARD_LIGHT, corner_radius=18); c3.grid(row=1, column=0, sticky="nsew", padx=(0,15), pady=10); c3.pack_propagate(False)
    ctk.CTkLabel(c3, text="Awareness Check", font=("Arial", 16), text_color=COLOR_SIDEBAR).pack(anchor="nw", padx=25, pady=(25,5))
    lbl_awareness = ctk.CTkLabel(c3, text="--", font=("Georgia", 42, "bold"), text_color=COLOR_SIDEBAR); lbl_awareness.pack(anchor="nw", padx=25)
    
    c4 = ctk.CTkFrame(grid, fg_color=COLOR_CARD_MID, corner_radius=18); c4.grid(row=1, column=1, sticky="nsew", padx=(15,0), pady=10); c4.pack_propagate(False)
    ctk.CTkLabel(c4, text="People helped", font=("Arial", 16), text_color="#F3EBDD").pack(anchor="nw", padx=25, pady=(25,5))
    lbl_helped = ctk.CTkLabel(c4, text="--", font=("Georgia", 42, "bold"), text_color="white"); lbl_helped.pack(anchor="nw", padx=25)
    
    # Lower Panels
    p_left = ctk.CTkFrame(grid, fg_color=COLOR_CARD_DARK, corner_radius=18); p_left.grid(row=2, column=0, sticky="nsew", padx=(0,15), pady=(15,0)); p_left.pack_propagate(False)
    ctk.CTkLabel(p_left, text="Unsure about a situation ?", font=("Georgia", 26, "bold"), text_color="#F3EBDD").pack(pady=(35,10))
    ctk.CTkLabel(p_left, text="Use our AI-driven analysis tool to identify signs\nof exploitation of forced labour.", font=("Arial", 15), text_color="#D9C2A3", justify="center").pack(pady=(0,25))
    ctk.CTkButton(p_left, text="Start Risk Assessment", fg_color=COLOR_CARD_ORANGE, font=("Arial", 15, "bold"), height=45, corner_radius=10).pack()
    
    p_right = ctk.CTkFrame(grid, fg_color=COLOR_PANEL_BG, corner_radius=18, border_width=1, border_color="#D9C2A3"); p_right.grid(row=2, column=1, sticky="nsew", padx=(15,0), pady=(15,0)); p_right.pack_propagate(False)
    ctk.CTkLabel(p_right, text="Recent Reports", font=("Georgia", 22, "bold"), text_color=COLOR_TEXT_DARK).pack(anchor="nw", padx=25, pady=(25,10))
    feed = ctk.CTkFrame(p_right, fg_color="transparent"); feed.pack(fill="both", expand=True, padx=25)
    
    # Update data dynamically
    data = fetch_dashboard_metrics()
    lbl_total.configure(text=data["total"])
    lbl_highrisk.configure(text=data["high_risk"])
    lbl_awareness.configure(text=data["awareness"])
    lbl_helped.configure(text=data["helped"])
    for report in data["recent"]:
        lbl = ctk.CTkLabel(feed, text=f"• {report}", font=("Arial", 14), text_color=COLOR_TEXT_DARK, anchor="w")
        lbl.pack(fill="x", pady=6)
        ctk.CTkFrame(feed, height=1, fg_color="#D9C2A3").pack(fill="x", pady=(2,6))


def load_insights_view():
    clear_workspace()
    update_sidebar_highlights("Insights")
    
    # Header Section
    ctk.CTkLabel(main_frame, text="Analytics & Insights", font=("Georgia", 32, "bold"), text_color=COLOR_TEXT_DARK).pack(anchor="nw", padx=40, pady=(25, 5))
    ctk.CTkLabel(main_frame, text="Review underlying vulnerability trends and systemic data distributions.", font=("Arial", 14), text_color="#5E4B3C").pack(anchor="nw", padx=40, pady=(0, 20))
    
    # Split Layout Grid Frame
    split_grid = ctk.CTkFrame(main_frame, fg_color="transparent")
    split_grid.pack(fill="both", expand=True, padx=40, pady=(0, 30))
    split_grid.columnconfigure(0, weight=1) # Left Panel (Takeaways + Donut)
    split_grid.columnconfigure(1, weight=1) # Right Panel (Horizontal Bar)
    split_grid.rowconfigure(0, weight=1)
    
    data = fetch_insights_data()
    theme_colors = [COLOR_CARD_ORANGE, COLOR_CARD_DARK, COLOR_CARD_MID, COLOR_CARD_LIGHT, "#A44E25", "#766555"]
    
    #LEFT COLUMN: KEY TAKEAWAYS + DONUT CHART BELOW IT
    
    left_panel = ctk.CTkFrame(split_grid, fg_color=COLOR_PANEL_BG, corner_radius=18, border_width=1, border_color="#D9C2A3")
    left_panel.grid(row=0, column=0, sticky="nsew", padx=(0, 15))
    
    ctk.CTkLabel(left_panel, text="System Key Takeaways", font=("Georgia", 18, "bold"), text_color=COLOR_TEXT_DARK).pack(anchor="nw", padx=20, pady=(15, 10))
    
    for insight in data["text_insights"]:
        box = ctk.CTkFrame(left_panel, fg_color=COLOR_WORKSPACE_BG, corner_radius=8)
        box.pack(fill="x", padx=20, pady=4)
        lbl = ctk.CTkLabel(box, text=insight, font=("Arial", 12), text_color=COLOR_TEXT_DARK, justify="left", wraplength=420)
        lbl.pack(anchor="w", padx=12, pady=8)

    # Donut Container now matches the warm panel color smoothly
    donut_container = ctk.CTkFrame(left_panel, fg_color=COLOR_PANEL_BG, corner_radius=14)
    donut_container.pack(fill="both", expand=True, padx=20, pady=(10, 15))
    
    # Set figure backgrounds to match COLOR_PANEL_BG
    fig_donut, ax_donut = plt.subplots(figsize=(4.5, 3.5), facecolor=COLOR_PANEL_BG)
    ax_donut.set_facecolor(COLOR_PANEL_BG)
    
    wedges, texts, autotexts = ax_donut.pie(
        data["counts"], 
        autopct='%1.0f%%',
        startangle=90,
        colors=theme_colors,
        pctdistance=0.72,
        textprops=dict(color="white", fontsize=10, weight="bold"),
        wedgeprops=dict(width=0.38, edgecolor=COLOR_PANEL_BG, linewidth=2.5) # Blends seamless edges
    )
    
    # Text adjustment loop for dark colors
    for i, autotext in enumerate(autotexts):
        if data["categories"][i] in ["Child Labour", "Wage Withholding", "Sexual Exploitation"]:
            autotext.set_color(COLOR_TEXT_DARK)
            
    ax_donut.set_title("Percentage Breakdown", fontfamily="sans-serif", fontsize=12, fontweight="bold", color=COLOR_TEXT_DARK, pad=5)
    
    ax_donut.legend(
        wedges, data["categories"],
        loc="center left",
        bbox_to_anchor=(0.95, 0.5),
        frameon=False,
        fontsize=9,
        labelcolor=COLOR_TEXT_DARK
    )
    
    plt.figure(fig_donut.number)
    plt.tight_layout()
    
    canvas_donut = FigureCanvasTkAgg(fig_donut, master=donut_container)
    canvas_donut.get_tk_widget().pack(fill="both", expand=True)
    canvas_donut.draw()


    #RIGHT COLUMN: DYNAMIC HORIZONTAL BAR CHART
    
    # Setting right panel frame container background to match color palette
    right_panel = ctk.CTkFrame(split_grid, fg_color=COLOR_PANEL_BG, corner_radius=18, border_width=1, border_color="#D9C2A3")
    right_panel.grid(row=0, column=1, sticky="nsew", padx=(15, 0))
    
    # Set figure background to match COLOR_PANEL_BG
    fig_bar, ax_bar = plt.subplots(figsize=(6, 6), facecolor=COLOR_PANEL_BG)
    ax_bar.set_facecolor(COLOR_PANEL_BG)
    
    bars = ax_bar.barh(data["categories"], data["counts"], color=COLOR_CARD_ORANGE, edgecolor=COLOR_CARD_DARK, height=0.55)
    
    # Border adjustments
    ax_bar.spines['top'].set_visible(False)
    ax_bar.spines['right'].set_visible(False)
    ax_bar.spines['left'].set_color(COLOR_TEXT_DARK)
    ax_bar.spines['bottom'].set_color(COLOR_TEXT_DARK)
    
    # Axis dynamic text mapping adjustments to theme text rules
    ax_bar.tick_params(colors=COLOR_TEXT_DARK, labelsize=11)
    ax_bar.set_title("Volume of Incidents by Category", fontfamily="sans-serif", fontsize=14, fontweight="bold", color=COLOR_TEXT_DARK, pad=15)
    
    # Numbers rendered cleanly on the bars
    for bar in bars:
        width = bar.get_width()
        ax_bar.annotate(f' {int(width)}', xy=(width, bar.get_y() + bar.get_height() / 2), xytext=(3, 0),
                        textcoords="offset points", ha='left', va='center', fontsize=11, color=COLOR_TEXT_DARK, weight='bold')
    
    plt.figure(fig_bar.number)
    plt.tight_layout()
    
    canvas_bar = FigureCanvasTkAgg(fig_bar, master=right_panel)
    canvas_bar.get_tk_widget().pack(fill="both", expand=True, padx=20, pady=20)
    canvas_bar.draw()


#SIDEBAR LOGIC & GENERATION

sidebar = ctk.CTkFrame(root, width=260, fg_color=COLOR_SIDEBAR, corner_radius=0)
sidebar.pack(side="left", fill="y")
sidebar.pack_propagate(False)

logo = ctk.CTkLabel(sidebar, text="SafePath", font=("Georgia", 32, "bold"), text_color="white")
logo.pack(pady=(40, 30), anchor="w", padx=30)

sidebar_buttons = {}
menu_items = ["Dashboard", "Situation Check", "Report Case", "Records", "Insights", "Resources", "Echoes of Home", "Settings"]

def update_sidebar_highlights(active_page):
    for name, button in sidebar_buttons.items():
        if name == active_page:
            button.configure(fg_color="#C47A3A", font=("Arial", 14, "bold"))
        else:
            button.configure(fg_color="transparent", font=("Arial", 14, "normal"))

for item in menu_items:
    if item == "Dashboard":
        cmd = load_dashboard_view
    elif item == "Insights":
        cmd = load_insights_view
    else:
        cmd = None
        
    btn = ctk.CTkButton(
        sidebar, text=item, command=cmd, hover_color="#8A4F27",
        text_color="white", anchor="w", height=40
    )
    btn.pack(fill="x", padx=20, pady=5)
    sidebar_buttons[item] = btn

tagline = ctk.CTkLabel(sidebar, text="Awareness. Protection. Freedom.", font=("Arial", 11, "italic"), text_color="#D9C2A3")
tagline.pack(side="bottom", pady=25)

# Initialize view application
main_frame.pack(side="right", fill="both", expand=True)
load_dashboard_view()

root.mainloop()

ModuleNotFoundError: No module named 'psycopg2'